In [ ]:
use database TRANSFORM_ENGCA_DEV;
use schema DBT_JLOGAN_ETHELO;
use role TRANSFORMER_ENGCA_DEV;

## 1) Create Table of Context Primers for Each Target

### Embedding Model Context Priming
This SQL query creates and populates the `target_context_primers` table. Storing these Large Language Model (LLM)-generated context primers for each survey topic ensures consistent input for generating comment embeddings, supporting reproducibility if re-runs are needed. This context priming strategy establishes a controlled semantic baseline by highlighting broad themes common across all comments; this helps the resulting embeddings de-emphasize these general variations and more clearly represent each comment's specific content and distinct perspective.

Note: This specific context priming part was a carry-over from previous work and may not be necessary. Future work should evaluate the effectiveness of this addition to the global context priming. 

In [ ]:
-- This query aims to generate a specific "context primer" for each unique 'TARGET' category found in the comments.
-- It first aggregates comments related to each TARGET, then uses these aggregated comments
-- to prompt a Large Language Model (LLM) via Snowflake Cortex to create a ~150-word primer.
-- The primer is intended to summarize common sub-topics factually, without including opinions.

Create or replace table target_context_primers  as 
-- CTE 1: target_comment_agg
-- Purpose: Groups comments by the 'TARGET' field and concatenates them into a single
--          string for each TARGET, to be used as input for the LLM.
WITH target_comment_agg AS (
    SELECT 
        TARGET,
        LISTAGG(SUBSTRING(CONTENT, 1, 500), '; ') AS topic_comments  -- Combines comments (max 500 chars each) into one string.
    FROM TRANSFORM_ENGCA_PRD.ANALYTICS.STG_COMMENTS as a
    GROUP BY TARGET
),

-- CTE 2: context_primer_raw
-- Purpose: For each TARGET, this CTE constructs a detailed prompt and calls the 
--          Snowflake Cortex LLM ('claude-3-5-sonnet') to generate the context primer.
--          The prompt guides the LLM to identify common sub-topics from 'topic_comments'
--          and create a neutral, factual summary of about 150 words, excluding opinions and introductions.
context_primer_raw AS (
    SELECT
        TARGET,
        topic_comments, 
        SNOWFLAKE.CORTEX.COMPLETE('claude-3-5-sonnet', 
            CONCAT(
                -- Instructions and context for the LLM:
                'You will receive a semi-colon separated list of user comments. These comments are from people impacted by the January 2025 Southern California wildfires (specifically the Palisades Fire in Pacific Palisades and the Eaton Fire in Altadena, Los Angeles County). ',
                'All these comments pertain to a specific Overall Topic, which is: "', TARGET, '". ',
                'Your objective is to distill these Comments to identify the principal subjects of focus and frequently mentioned themes that appear in multiple contributions related to this Overall Topic. ',
                'From these, construct a neutral and descriptive context primer of approximately 150 words that highlights these common subjects of discussion. ',
                'The primer should map out the landscape of commonly addressed points factually, without elaborating on the varying stances or arguments presented by the commenters on those points. Importantly, your response should contain the context primer and nothing else (e.g. no introduction).',
                -- The aggregated comments are appended to the prompt:
                'The comments to analyze are:\n', topic_comments
            )
        ) AS context_primer                         -- The output from the LLM (typically a JSON object) is stored in this column.
    FROM target_comment_agg
)

-- Final SELECT Statement
-- Purpose: Retrieves the TARGET, the generated 'context_primer' output from the LLM,
SELECT
    TARGET,
    context_primer,                                 
    LENGTH(context_primer) AS cntxt_prmr_len
FROM context_primer_raw;


In [ ]:
select * from target_context_primers

## 2) Create Table of Embeddings and an LLM Summary For Each Comment 

The embeddings leverage the context priming and the LLM summary is used for cluster visualization testing in streamlit

**IMPORTANT:** It appears that re-running this will not change the embedding values as long as the source text does not change (e.g. the previous code block is run). 

In [ ]:
Create or replace table target_embeddings_w_summary as
Select
    COMMENT_ID,
    TYPE,
    a.TARGET,
    CONTENT,
    -- Create text to be used for embedding. We construct a conditional context primer by combining: 
    -- 1) an universal wildfire introduction
    -- 2) the target specific context primer we made in the previous cell, and 
    -- 3) the original comment for nuanced semantic embedding relevant to that target.
    CONCAT(
        -- Part 1: Universal Wildfire Context Primer
        'This comment is from an individual impacted by the January 2025 Southern California wildfires. These devastating events included the human-caused Palisades Fire in Pacific Palisades and the suspected utility-linked Eaton Fire in Altadena (implicating entities like SoCal Edison/SCE). Fueled by severe Santa Ana winds and prolonged dry conditions, these fires resulted in fatalities, mass evacuations, and the widespread destruction of thousands of homes and community structures.  The extensive aftermath has involved significant economic hardship, profound emotional trauma, and generated multifaceted community discussions on necessary recovery paths and future preparedness.',
    
    -- Part 2: LLM-generated Context Primer specific to this comment's TARGET
    'The following provides further context for the specific survey topic of "', a.TARGET, '": ', b.context_primer, '\n\n', 
    
    -- Part 3:The User Comment
    'User Comment: Regarding this survey topic, the affected individual stated: "', a.CONTENT, '"'
) AS primed_comment_for_embedding,

SNOWFLAKE.CORTEX.EMBED_TEXT_1024('voyage-multilingual-2', primed_comment_for_embedding) as primed_comment_embedding,

SNOWFLAKE.CORTEX.COMPLETE(
  'claude-3-5-sonnet', 
  ARRAY_CONSTRUCT(    
    OBJECT_CONSTRUCT(
      'role', 'system',
      'content', $$You will be provided a user comment from a survey, which will be framed around a specific topic or issue. These comments are from people impacted by recent wildfires in Los Angeles County (specifically the January 2025 Eaton Fire in Altadena and the Palisades Fire in Pacific Palisades).

TASK: Your objective is to create a succinct summary of the user's core message or main points *specifically concerning the stated topic/issue*. Focus on capturing the essence of their perspective on this topic, particularly as it relates to the wildfire's impact, recovery, prevention, or associated challenges. Your response must be ONLY the summarized comment itself, and it must be 25 words or less. The summary must still sound like it is in the user's own voice. If the comment is already 25 words or less, you can just return the comment itself.$$
    ),
    OBJECT_CONSTRUCT(
      'role', 'user',
      'content', 'Regarding the issue of ' || a.TARGET || ', an affected individual offered this perspective: "' || CONTENT || '"'
    )
  ),
  OBJECT_CONSTRUCT(   
    'temperature', 0.05,
    'max_tokens', 40 -- Approx. 30 words is ~40 tokens.
  )
)['choices'][0]['messages']::varchar AS llm_summary
FROM TRANSFORM_ENGCA_PRD.ANALYTICS.STG_COMMENTS as a
inner join target_context_primers as b on a.TARGET = b.TARGET;


In [ ]:
select COMMENT_ID, TARGET, CONTENT, LLM_SUMMARY, PRIMED_COMMENT_EMBEDDING from target_embeddings_w_summary

### Get data ready for topic modeling

In [ ]:
# Import python packages
import pandas as pd
import numpy as np
import os

from snowflake.snowpark.context import get_active_session
session = get_active_session()

# set environment variables to get bertopic to work in this environment
os.environ["NUMBA_CACHE_DIR"] = "/tmp"
os.environ["TRANSFORMERS_CACHE"] = "/tmp"

cmnt_embeddings_df = session.sql('''
Select
    COMMENT_ID,
    TARGET,
    CONTENT,
    LLM_SUMMARY,     
    PRIMED_COMMENT_EMBEDDING
FROM target_embeddings_w_summary
''').to_pandas()

full_embedding = np.array(cmnt_embeddings_df['PRIMED_COMMENT_EMBEDDING'].tolist())
docs = np.array(cmnt_embeddings_df['CONTENT'].tolist())
docs_summary = np.array(cmnt_embeddings_df['LLM_SUMMARY'].tolist())
categories = cmnt_embeddings_df['TARGET'].tolist()


# Automatically optimize topic modeling hyperparameters
This code runs a topic modeling loop to optimize for hyperparameters that yield high cluster separation and distinction. The resulting hyperparameters can be copy/pasted in the configuration blocks below for validation. Note that for larger datasets, running many trials may be prohibitive in the Snowflake environment.

NOTE: this code is currently configured to optimize for one grouping of comment categories at at time. The user is expected to filter embeddings to a specific category before running the optimization.

In [ ]:
import time
import optuna
import numpy as np
from bertopic import BERTopic
from umap import UMAP
from hdbscan import HDBSCAN
from sklearn.metrics import silhouette_score, calinski_harabasz_score
from sklearn.feature_extraction.text import CountVectorizer

# Disable printing a message for each trial
optuna.logging.set_verbosity(optuna.logging.WARNING)


# Returns higher values when the input number of topics matches what we
# want/expect from our data. This should be configured per comment category!
def score_topic_count(num_topics: int) -> float:
    if num_topics == 4:
        return 1 # we want this many topics
    # elif num_topics in (3,5):
    #     return 1 # this is good, but slightly less desireable
    # elif num_topics in (2,6):
    #     return .5 # this is okay, but less desireable
    else:
        return 0


# Returns higher values when the number of classified documents is relatively high
# (but not too high)
def score_classified_percentage(classified_ratio: float) -> float:
    if .45 <= classified_ratio <= .90: # >90% classified means something went wrong, probably
        return 1.0
    elif .4 <= classified_ratio < .5:
        return .85
    elif .85 <= classified_ratio < .9:
        return .85
    # elif .9 <= classified_ratio <= 1:
    #     return .5
    else:
        return 0


# Normalize from (-1, 1) to (0, 1), where 1 is good
def normalize_silhouette_score(ss):
    return (ss + 1) / 2


# Normalize from (0, inf) to (0, 1). We use a sigmoid function to cap values at 1.
# Using a constant value of 5 is somewhat arbitrary, though most input score values
# are <10, so anything much higher might compress output values too much.
def normalize_calinski_harabasz_score(chs, C=5):
    return chs / (chs + C)     
        

In [ ]:
def objective(trial: optuna.trial.Trial):    
    # Set up parameters to optimize for
    umap_n_neighbors = trial.suggest_int('umap_n_neighbors', 3, 30)
    umap_n_components = trial.suggest_int('umap_n_components', 2, 6)
    umap_min_dist = trial.suggest_float('umap_min_dist', 0.0, 0.9)

    hdbscan_min_cluster_size = trial.suggest_int('hdbscan_min_cluster_size', 5, 10)
    hdbscan_min_samples = trial.suggest_int('hdbscan_min_samples', 2, 8)
    hdbscan_cluster_selection_epsilon = trial.suggest_float('hdbscan_cluster_selection_epsilon', 0.0, 0.8) # Note: only valid if method is 'eom'

    # Create models to execute with
    umap_model = UMAP(n_neighbors=umap_n_neighbors,
                      n_components=umap_n_components,
                      min_dist=umap_min_dist,
                      metric='cosine',
                      random_state=42) # For reproducibility within this trial

    hdbscan_model = HDBSCAN(min_cluster_size=hdbscan_min_cluster_size,
                            min_samples=hdbscan_min_samples,
                            cluster_selection_method='eom',
                            prediction_data=True,
                            gen_min_span_tree=True,
                            cluster_selection_epsilon=hdbscan_cluster_selection_epsilon)

    # This is the same vectorizer model we use when actually running topic modeling
    vectorizer_model = CountVectorizer(
        stop_words='english',
        min_df=0.01,
        max_df=0.5,
        ngram_range=(1, 2)
    )

    # Finally initialize topic modeler
    topic_model = BERTopic(
        top_n_words=1,
        hdbscan_model=hdbscan_model,
        umap_model=umap_model,
        vectorizer_model=vectorizer_model,
        verbose=False
    )

    # !!
    # !! Edit this to change the optimized category
    # !!
    category_indices = [i for i, cat in enumerate(categories) if cat == 'Debris removal & environmental recovery']

    # Extract documents and embeddings for this category
    # category_docs_full = np.array(docs)[category_indices]
    category_docs = np.array(docs_summary)[category_indices] # Ensure docs is indexable like a list or numpy array
    category_embeddings = full_embedding[category_indices]
    num_documents = len(category_docs)

    # Convert to numpy array to make BERTopic happy
    category_embeddings = np.array(category_embeddings)

    try:
        topics, _ = topic_model.fit_transform(list(category_docs), category_embeddings)
        topic_info = topic_model.get_topic_info()
    except Exception as e:
        # It's not uncommon to try and optimize invalid parameters for a given dataset. The best we
        # can do here is return a strong negative signal.
        return -2



    mask = [value != -1 for value in topics]
    assigned_fe = category_embeddings[mask]
    assigned_tp = np.array(topics)[mask]

    # Calculate clustering evaluation scores
    ss = silhouette_score(assigned_fe, assigned_tp)
    ss = normalize_silhouette_score(ss)

    chs = calinski_harabasz_score(assigned_fe, assigned_tp)
    chs = normalize_calinski_harabasz_score(chs)

    # Combine clustering evaluation scores into a "base" score
    base_score = (ss + chs) / 2
    
    # Calculate score scaling factors
    num_topics = len(topic_info[topic_info.Topic != -1])
    topic_count_score_val = score_topic_count(num_topics)

    num_classified_documents = np.sum(np.array(topics) != -1)
    classified_ratio = (num_classified_documents / num_documents)
    classified_percentage_score_val = score_classified_percentage(classified_ratio)

    final_score = base_score * topic_count_score_val * classified_percentage_score_val

    return final_score


start_time = time.time()
n_trials = 600
study = optuna.create_study(direction='maximize', study_name="bertopic_optimization")

study.optimize(objective, n_trials=n_trials)

print()
print(f'elapsed: {round(time.time()-start_time, 2)} sec')

print(f"Best trial number: {study.best_trial.number}")
print(f"Best value: {study.best_value:.4f}")
print("Best hyperparameters:")
for key, value in study.best_params.items():
    print(f"  {key}: {value}")
#Best value: 0.4819 

## Conduct Topic Modeling

## UMAP Parameters for Topic Modeling

These parameters control the initial dimensionality reduction before clustering. They determine how documents are arranged in lower-dimensional space for topic discovery.

### Parameters:

**`n_neighbors`** (default: 10)
- Lower values (3-10): Preserves more local structure, resulting in more granular, specific topics
- Higher values (15-50): Preserves more global structure, resulting in broader, more general topics

**`n_components`** (default: 5)
- Lower values (2-5): Forces more compression, may merge related topics
- Higher values (10-50): Preserves more information, allows for more nuanced topic separation

**`min_dist`** (default: 0.05)
- Lower values (0.0-0.1): Allows tighter clusters, better for finding distinct topics
- Higher values (0.5-0.99): Spreads points out more, may help if clusters are too tight

**General Tuning Strategy:** Decrease `n_neighbors` for more specific topics, increase `n_components` if topics are merging that shouldn't be.


In [ ]:
# --- UMAP Parameters for Topic Modeling ---
# Category-specific UMAP parameters
umap_topic_model_params = {
    'Emergency communication': {
        'n_neighbors': 10,
        'n_components': 5,
        'min_dist': 0.02,
    },
    'Debris removal & environmental recovery': {
        'n_neighbors': 21,
        'n_components': 5,
        'min_dist': 0.551285608966453,
        'metric': 'cosine'
    },


      
    # 'Debris removal & environmental recovery': {
    #     'n_neighbors': 9,
    #     'n_components': 6,
    #     'min_dist': 0.14506065160825884,
    # },

    'Infrastructure & utilities restoration': {
        'n_neighbors': 14,#20,
        'n_components': 5,
        'min_dist': 0.01,
        'metric': 'cosine',
    },
    'Emergency planning & community safety': {
        'n_neighbors': 12,
        'n_components': 18,
        'min_dist': 0.12985519154746097,
        #'metric': 'cosine',
    },
    'Housing & rebuilding': {
        'n_neighbors': 14,
        'n_components': 5,
        'min_dist': 0.2,
    },
    
    'Financial & legal assistance': {
        'n_neighbors': 15,
        'n_components': 10,
        'min_dist': 0.004232706079901838
    },
    'Climate & community resilience': {
        'n_neighbors': 12,
        'n_components': 17,
        'min_dist': 0.8090040354896819  ,
    },
    'Wildfire prevention prioritization & accountability': {
        'n_neighbors': 22,
        'n_components': 11,
        'min_dist': 0.5340317020748925,
    },


    
    # 'Prioritize recovery themes': {
    #     'n_neighbors': 10,
    #     'n_components': 5,
    #     'min_dist': 0.05,
    # },
    'Economic recovery & small business support': {
        'n_neighbors': 10,
        'n_components': 11,
        'min_dist': 0.15150412176614422,
    },
    # 'Emotional & mental health support ': {
    #     'n_neighbors': 33,
    #     'n_components': 10,
    #     'min_dist': 0.33972042681516623,
    # }
    'Emotional & mental health support ': {
        'n_neighbors': 19,
        'n_components': 7,
        'min_dist': 0.19152773135019338,
    }
}





# Default UMAP parameters for topic modeling if not specified for a category
default_umap_topic_model_params = {
    'n_neighbors': 5,
    'n_components': 10,     # Default number of components for topic modeling
    'min_dist': 0.05,
    'metric': 'euclidean',
    'random_state': 42
}

## HDBSCAN Parameters

These parameters control how topics are identified from the reduced embeddings. They determine the minimum size of topics and how conservative the clustering is.

### Parameters:

**`min_cluster_size`** (default: 8)
- Lower values (5-10): Creates more topics, including smaller niche topics
- Higher values (20-50): Creates fewer, larger topics; more documents marked as outliers

**`min_samples`** (default: 5)
- Lower values (1-5): More liberal clustering, fewer outliers
- Higher values (10+): More conservative clustering, more outliers
- Usually set to same value as `min_cluster_size` or slightly lower

**`cluster_selection_epsilon`** (default: 0)
- Diabled (0): No clusters will be merged after being created
- Lower values (0-.5): Clusters will be merged if they are very close together
- Higher values (.5+): Clusters that are further apart may be merged together

**General Tuning Strategy:** If too many documents are unassigned (outliers), decrease the `min_*` values. If topics are too small/specific, increase them. If there are too many small, distinct clusters that should be a part of the same group, increase `cluster_selection_epsilon`.


In [ ]:
# HDBSCAN parameters for each category
hdbscan_params = {
    'Emergency communication': {
        'min_cluster_size': 6,
        'min_samples': 5
    },
    'Debris removal & environmental recovery': {
        'min_cluster_size': 5,
        'min_samples': 4,#4,
        'cluster_selection_epsilon': 0.20824785767091733
    },

    # 'Debris removal & environmental recovery': {
    #     'min_cluster_size': 5,
    #     'min_samples': 4,#4,
    #     'cluster_selection_epsilon': 0.45804594638994695
    # },

    'Infrastructure & utilities restoration': {
        'min_cluster_size': 6,#10,
        'min_samples': 3,#8,
        #'allow_single_cluster': True,
    },
    'Emergency planning & community safety': {
        'min_cluster_size': 15,
        'min_samples': 2,
        'cluster_selection_epsilon': .4558529642413047
    },
    'Housing & rebuilding': {
        'min_cluster_size': 8,
        'min_samples': 4,
        #'cluster_selection_epsilon': .65
    },
    'Financial & legal assistance': {
        'min_cluster_size': 17,
        'min_samples': 2,
        'cluster_selection_epsilon': 0.19105758645971477,
    },
    'Climate & community resilience': {
        'min_cluster_size': 15,
        'min_samples': 2,
        'hdbscan_cluster_selection_epsilon': 0.20709624774541702
    },
    'Wildfire prevention prioritization & accountability': {
        'min_cluster_size': 9,
        'min_samples': 2,
        'hdbscan_cluster_selection_epsilon': 0.22857671175530273
    },
    # 'Prioritize recovery themes': {
    #     'min_cluster_size': 8,
    #     'min_samples': 5,
    #     'hdbscan_cluster_selection_epsilon': 0,
    # },


    
    'Economic recovery & small business support': {
        'min_cluster_size': 8,
        'min_samples': 3,
        'hdbscan_cluster_selection_epsilon': 0.08327763877585756,
    },
    # 'Emotional & mental health support ': {
    #     'min_cluster_size': 6,
    #     'min_samples': 3,
    #     'hdbscan_cluster_selection_epsilon': 0.3805211678372587,
    # }
    'Emotional & mental health support ': {
        'min_cluster_size': 6,
        'min_samples': 4,
        'hdbscan_cluster_selection_epsilon': 0.3747802543340463,
    }
}



# Default HDBSCAN parameters if not specified for a category
default_hdbscan_params = {
    'min_cluster_size': 8,
    'min_samples': 5,
    'metric': 'euclidean', # HDBSCAN often works on the UMAP output, so euclidean is common here
    'cluster_selection_method': 'eom', # Excess of Mass algorithm for cluster selection
    'prediction_data': True, # Needed if you want to predict clusters for new data points
    'cluster_selection_epsilon': 0,
    'allow_single_cluster': False,
}

## UMAP Parameters for 2D Visualization

These parameters control the 2D scatter plot visualization of topics. This supervised UMAP uses topic assignments to create visual clusters.

### Parameters:

**`n_neighbors`** (default: 20)
- Lower values (5-15): Topics appear as tight, distinct clusters
- Higher values (30-100): Better shows relationships between topics but clusters may overlap

**`min_dist`** (default: 0.8)
- Lower values (0.1-0.5): Points pack tightly, clear cluster boundaries
- Higher values (0.8-0.99): Points spread out more, easier to see individual documents

**`target_weight`** (default: 0.2)
- 0.0 = unsupervised, 1.0 = fully supervised
- Lower values (0.1-0.3): More organic layout based on embedding similarity
- Higher values (0.5-0.9): Forces same-topic documents together

**General Tuning Strategy:** Increase `min_dist` if points overlap too much, increase `target_weight` if topics don't separate well visually.

In [ ]:

# --- UMAP Parameters for 2D Visualization ---
# Category-specific parameters for the 2D UMAP visualization
umap_visualization_params = {
    'Emergency communication': {
        'n_neighbors': 10,
        'target_weight': 0.3,
        'min_dist': 0.2,
    },
    'Debris removal & environmental recovery': {
        'n_neighbors': 80,
        'target_weight': .999,
        'min_dist': 0.1,
    },
    'Infrastructure & utilities restoration': {
        'n_neighbors': 10,
        'target_weight': 0.3,
        'min_dist': 0.5,
    },
    'Emergency planning & community safety': {
        'n_neighbors': 10,
        'target_weight': 0.5,
        'min_dist': 0.6,
    },
    'Housing & rebuilding': {
        'n_neighbors': 10,
        'target_weight': 0.5,
        'min_dist': 0.2,
    },
    'Financial & legal assistance': {
        'n_neighbors': 10,
        'target_weight': 0.5,
        'min_dist': 0.5,
    },
    
    'Climate & community resilience': {
        'n_neighbors': 30,
        'target_weight': 0.9,
        'min_dist': 0.3,
    },
    
    'Wildfire prevention prioritization & accountability': {
        'n_neighbors': 30,
        'target_weight': 0.7,
        'min_dist': 0.8,
    },



    
    # 'Prioritize recovery themes': {
    #     'n_neighbors': 10,
    #     'target_weight': 0.5,
    #     'min_dist': 0.5,
    # },
    'Economic recovery & small business support': {
        'n_neighbors': 20,
        'target_weight': 0.85,
        'min_dist': 0.55,
    },
    'Emotional & mental health support ': {
        'n_neighbors': 40,
        'target_weight': 1,
        'min_dist': 0.001,
    }
}

# Default UMAP parameters for visualization
default_umap_visualization_params = {
    'n_neighbors': 20,
    'min_dist': 0.5,
    'target_weight': 0.2,
    'metric': 'euclidean',
    'target_metric': 'categorical',
    'random_state': 50
}

In [ ]:
from umap import UMAP
from bertopic import BERTopic
import pandas as pd
import numpy as np
import json
import os
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer
    
# Create a custom CountVectorizer with stop words removal and document frequency parameters
vectorizer_model = CountVectorizer(
    stop_words='english',           # Remove English stop words
    min_df=0.01,                    # Ignore terms that appear in less than X% of documents
    max_df=0.5,                     # Ignore terms that appear in more than X% of documents
    ngram_range=(1, 2)              # Consider both unigrams and bigrams for more meaningful phrases
)

# Dictionary to store results
category_topics = {}
all_topics_df = pd.DataFrame()  # Will hold all topics across categories


# Iterate through each unique category
unique_categories = list(set(categories)) # Ensure 'categories' is defined
print(f"Found {len(unique_categories)} unique categories")

for category in unique_categories:
    # Skip None category
    if category is None or category == 'None':
        print("Skipping None category")
        continue
        
    # Filter data for this category
    category_indices = [i for i, cat in enumerate(categories) if cat == category]
    if category not in [
        'Emergency communication',
        'Debris removal & environmental recovery',
        'Infrastructure & utilities restoration',
        'Emergency planning & community safety',
        'Housing & rebuilding',
        'Financial & legal assistance',
        'Climate & community resilience',
        'Wildfire prevention prioritization & accountability',
        'Economic recovery & small business support',
        'Emotional & mental health support ',
    ]:
        continue

    print(f"\nProcessing category: {category} ({len(category_indices)} comments)")
    
    # Skip if too few documents
    min_docs_for_processing = max(default_umap_topic_model_params.get('n_neighbors', 5), 
                                  default_hdbscan_params.get('min_cluster_size', 5), 
                                  10) # Ensure enough samples for UMAP and HDBSCAN
    
    # Adjust min_docs if category specific UMAP params exist
    if category in umap_topic_model_params:
        min_docs_for_processing = max(min_docs_for_processing, umap_topic_model_params[category].get('n_neighbors',5))
    if category in hdbscan_params:
         min_docs_for_processing = max(min_docs_for_processing, hdbscan_params[category].get('min_cluster_size',5))

    
    if len(category_indices) < min_docs_for_processing:
        print(f"Skipping category '{category}' - not enough documents ({len(category_indices)}). Needs at least {min_docs_for_processing}.")
        continue
    
    # Extract documents and embeddings for this category
    category_docs = np.array(docs_summary)[category_indices] # Ensure docs is indexable like a list or numpy array
    category_embeddings = full_embedding[category_indices]
    category_viz_embeddings = full_embedding[category_indices]

    # Convert to numpy array to make BERTopic happy
    category_embeddings = np.array(category_embeddings)
    category_viz_embeddings = np.array(category_viz_embeddings)
    
    # Get UMAP parameters for topic modeling for this category
    current_umap_params = default_umap_topic_model_params.copy()
    if category in umap_topic_model_params:
        current_umap_params.update(umap_topic_model_params[category])
    
    #print(f"Using UMAP parameters for topic modeling in '{category}': "
    #      f"n_neighbors={current_umap_params['n_neighbors']}, "
    #      f"n_components={current_umap_params['n_components']}, "
    #      f"min_dist={current_umap_params['min_dist']}")

    # Create UMAP model with n components for topic modeling
    umap_model = UMAP(
        n_neighbors=current_umap_params['n_neighbors'],
        n_components=current_umap_params['n_components'],
        min_dist=current_umap_params['min_dist'],
        metric=current_umap_params['metric'],
        random_state=current_umap_params['random_state']
    )
    
    # Get HDBSCAN parameters for this category or use defaults
    category_hdbscan_params = default_hdbscan_params.copy()
    if category in hdbscan_params:
        category_hdbscan_params.update(hdbscan_params[category])
        
    #print(f"Using HDBSCAN parameters for '{category}': "
    #      f"min_cluster_size={category_hdbscan_params['min_cluster_size']}, "
    #      f"min_samples={category_hdbscan_params['min_samples']}")
    
    # Create HDBSCAN clustering model
    hdbscan_model = HDBSCAN(
        min_cluster_size=category_hdbscan_params['min_cluster_size'],
        min_samples=category_hdbscan_params['min_samples'],
        #metric=category_hdbscan_params['metric'],
        #cluster_selection_method=category_hdbscan_params['cluster_selection_method'],
        cluster_selection_epsilon=category_hdbscan_params['cluster_selection_epsilon'],
        prediction_data=category_hdbscan_params['prediction_data'],
        #allow_single_cluster=category_hdbscan_params['allow_single_cluster'],
        gen_min_span_tree=True,
    )
        
    # Create BERTopic model
    topic_model = BERTopic(
        top_n_words=10,
        hdbscan_model=hdbscan_model,
        umap_model=umap_model,
        vectorizer_model=vectorizer_model,
        verbose=False
    )

    if False:
        print(repr(umap_model))
        print()
        print(repr(hdbscan_model))
    
    # Fit BERTopic with embeddings
    try:
        topics, probs = topic_model.fit_transform(list(category_docs), category_embeddings) # Pass docs as list
    except Exception as e:
        print(f"Error during BERTopic fit_transform for category '{category}': {e}")
        print(f"Number of documents: {len(category_docs)}, Embeddings shape: {category_embeddings.shape}")
        print(f"UMAP params: {current_umap_params}")
        print(f"HDBSCAN params: {category_hdbscan_params}")
        continue # Skip to next category if BERTopic fails

    # Create topic info dataframe
    topic_info = topic_model.get_topic_info()
    print(f"Found {len(topic_info) -1 if not topic_info.empty and topic_info.iloc[0]['Topic'] == -1 else len(topic_info)} actual topics (excluding outliers) for category '{category}'")


    # Count how many documents were assigned to each topic
    unique_topic_ids, topic_doc_counts = np.unique(topics, return_counts=True)
    #print("Topic distribution:")
    for topic_id, count in zip(unique_topic_ids, topic_doc_counts):
        if topic_id != -1:
            print(f"  Topic {topic_id}: {count} documents")
    if -1 in unique_topic_ids:
        print(f"  Unassigned (-1): {list(topics).count(-1)} documents")
    else:
        print(f"  Unassigned (-1): 0 documents")

        
    # Add category column
    topic_info['CATEGORY'] = category
    
    # Create a unique prefix for topics
    safe_category = "".join(filter(str.isalnum, str(category)))[:5] if category else "unk" # Make safer prefix
    cat_prefix = f"{safe_category}_{len(all_topics_df)}_" # len(all_topics_df) might not be ideal if parallel, but ok sequentially
    
    topic_info['ORIGINAL_TOPIC'] = topic_info['Topic']
    topic_info['Topic'] = topic_info['Topic'].apply(lambda x: f"{cat_prefix}{x}" if x != -1 else f"{cat_prefix}outlier")
    
    # Add to the combined dataframe
    all_topics_df = pd.concat([all_topics_df, topic_info])
    
    # Create dataframe with documents and their topics
    doc_df = pd.DataFrame({
        'COMMENT_ID': [cmnt_embeddings_df.iloc[idx]['COMMENT_ID'] for idx in category_indices[:len(topics)]],
        'SUMMARY': category_docs[:len(topics)],
        'CATEGORY': category,
        'TOPIC': [f"{cat_prefix}{t}" if t != -1 else f"{cat_prefix}outlier" for t in topics[:len(category_docs)]],
        'ORIGINAL_TOPIC': topics[:len(category_docs)],
        'CONTENT': [cmnt_embeddings_df.iloc[idx]['CONTENT'] for idx in category_indices[:len(topics)]],
    })
    
    # Now create a supervised 2D UMAP using the topic assignments as labels
    
    # Use the original topic numbers as labels, but set -1 topics to None for supervised UMAP
    topic_labels = np.array(topics[:len(category_docs)])
    
    # Use the original topic numbers (-1 for unassigned) as labels for supervised UMAP
    # Ensure it's an integer array, as UMAP expects this for categorical targets.
    topic_labels_for_umap = np.array(topics[:len(category_docs)], dtype=np.int_) # Using np.int_ or np.int32 etc.
    
    # Get UMAP visualization parameters for this category
    visualization_params = default_umap_visualization_params.copy()
    if category in umap_visualization_params:
        visualization_params.update(umap_visualization_params[category])
    
    #print(f"Using UMAP parameters for visualization in '{category}': "
    #      f"n_neighbors={visualization_params['n_neighbors']}, "
    #      f"min_dist={visualization_params['min_dist']}, "
    #      f"target_weight={visualization_params['target_weight']}")
    
    # Create supervised UMAP model for visualization
    umap_model_2d = UMAP(
        n_neighbors=visualization_params['n_neighbors'],
        n_components=2,
        min_dist=visualization_params['min_dist'],
        metric=visualization_params['metric'],
        target_metric=visualization_params['target_metric'],
        target_weight=visualization_params['target_weight'],
        random_state=visualization_params['random_state']
    )
    
    # Handle the case where all documents might be unassigned (-1)
    if np.all(topic_labels == -1):
        print("All documents are unassigned. Using unsupervised UMAP for visualization.")
        umap_2d_embeddings = umap_model_2d.fit_transform(category_viz_embeddings)
    else:
        # Fit supervised UMAP with topic labels
        umap_2d_embeddings = umap_model_2d.fit_transform(category_viz_embeddings, y=topic_labels_for_umap)
    
    # Add UMAP coordinates
    doc_df['UMAP_1'] = [coord[0] for coord in umap_2d_embeddings[:len(doc_df)]]
    doc_df['UMAP_2'] = [coord[1] for coord in umap_2d_embeddings[:len(doc_df)]]
    
    # Store in category_topics dictionary
    category_topics[category] = {
        'model': topic_model,
        'doc_df': doc_df,
        'topic_info': topic_info
    }

# Convert all topic info to strings for Snowflake
for col in all_topics_df.columns:
    if all_topics_df[col].dtype == 'object':
        all_topics_df[col] = all_topics_df[col].astype(str)

# Fill NaN values (this does nothing?)
all_topics_df = all_topics_df.fillna("None")


# Create combined documents dataframe
all_docs_df = pd.concat([cat_data['doc_df'] for cat_data in category_topics.values()])

session.write_pandas(
    all_topics_df,
    table_name="COMMENT_TARGET_TOPIC_INFO",
    #database="TRANSFORM_ENGCA_DEV",
    #schema="DBT_JLOGAN_ETHELO",
    overwrite=True,
    quote_identifiers=False,
    auto_create_table=True
)

print("Writing documents with topics to Snowflake...")
session.write_pandas(
    all_docs_df,
    table_name="COMMENT_TARGET_DOCS_WITH_TOPICS",
    #database="TRANSFORM_ENGCA_DEV",
    #schema="DBT_JLOGAN_ETHELO",
    overwrite=True,
    quote_identifiers=False,
    auto_create_table=True
)
print("Topic modeling complete!")

# Apply Topic Labels
Topics are grouped together using the above cell, but we want to generate a label for each group to be able to summarize what topic it represents.

In [ ]:
-- Generate LLM-based topic descriptions using Snowflake Cortex
CREATE OR REPLACE TABLE TARGET_COMMENT_TOPIC_DESCRIPTIONS AS


WITH topic_comment_agg AS (
    SELECT 
        a.TOPIC,
        a.CATEGORY,
        COUNT(*) AS n,
        LISTAGG(SUBSTRING(CONTENT, 1, 500), '; ') AS topic_comments
        ,b.representation
    from COMMENT_TARGET_DOCS_WITH_TOPICS as a
    left join COMMENT_TARGET_TOPIC_INFO as b on a.topic = b.topic
    where a.original_topic >= 0
    GROUP BY a.TOPIC, a.CATEGORY,REPRESENTATION
),

topic_desc_raw AS (
    SELECT
        topic,
        category,
        n,
        LENGTH(topic_comments) AS n_char,
        topic_comments,
        -- use 'claude-3-5-sonnet' for final labels. Use 'llama4-maverick' for testing. 
        SNOWFLAKE.CORTEX.COMPLETE('claude-3-5-sonnet', --llm model to use
            CONCAT($$The following is a semi-colon separated list of comments from people impacted by recent wildfires in Los Angeles county (the Eaton Fire in Altadena and the Palisades fire in Pacific Palisades). For these comments, please provide a brief topic title. The keywords associated with these comments and this topic are:$$,representation, $$. The topic title should be from 2 to 4 words and should be extremely specific and descriptive of the throughline concern addressed by the comments. It is crucial that the title be representative of every single comment you are provided. Your response should contain the 2-4 word topic title and nothing else. Only use alphanumeric characters and spaces. No quotes.  <comments>$$, 
            topic_comments, '</comments>')) AS desc_raw
    FROM topic_comment_agg
)

SELECT
    topic,
    category,
    n,
    desc_raw AS title
FROM topic_desc_raw;


In [ ]:
select * from TARGET_COMMENT_TOPIC_DESCRIPTIONS

### Export note:

Below is the query used to generate the table for the export CSV, one category at a time. We prefer to use the query in worksheet rather than here since when exporting from a notebook, Snowflake unnecessarily saves an index/row number along with the data.

In [ ]:
select
    init_embeds.comment_id,
    init_embeds.target as category,
    coalesce(topic_desc.title, 'Other') as subcategory,
    REGEXP_REPLACE(
        -- replace "smart" single quotes with normal single quotes
        REGEXP_REPLACE(init_embeds.content, '[\u2018\u2019]', $$'$$),
        -- replace "smart" double quotes with normal double quotes
        '[\u201c\u201d]', '"'
    ) as content,
    topics.umap_1,
    topics.umap_2,
from target_embeddings_w_summary init_embeds
left join COMMENT_TARGET_DOCS_WITH_TOPICS topics on init_embeds.comment_id = topics.comment_id
left join TARGET_COMMENT_TOPIC_DESCRIPTIONS topic_desc on topics.topic = topic_desc.topic
where target = 'Debris removal & environmental recovery'
order by init_embeds.comment_id
;